Importing the necessary packages to create the model:

In [1]:
import tensorflow as tf
from tensorflow.math import sigmoid
from tensorflow import keras

from keras.models import Sequential
from keras.layers import Conv2D
from keras.layers import MaxPooling2D
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Dropout

from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import json

import scalogram_cnn_project.settings.config as config
from scalogram_cnn_project.utils.list_files import list_files

2026-03-06 17:01:36.339439: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 17:01:40.430414: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-06 17:01:44.723077: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Below, we define the seed utilized for the dropout layers and the value of some hyperparameters that can be later optimized:

In [2]:
SAVE_MODEL = False

master_model_name = "my_model"

master_seed = 42 

os.environ["PYTHONHASHSEED"] = str(master_seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
np.random.seed(master_seed)
tf.random.set_seed(master_seed)
tf.config.experimental.enable_op_determinism()

## Parameters for which to try a gridsearch
master_learning_rate = 3e-5
master_batch_size = 16
master_epsilon = 1e-3

## Parameter that is only important when making inferences with the CNN
master_momentum = 0.99

Creating a function to import scalograms from the training and the test datasets, already normalizing the data so that the intensity of each pixel lies between 0 and 1:

In [3]:
def load_data(folder="GeneratedScalogramsGrayMoreOverlap",
              channels=["C3", "C4"],
              cmap="viridis"):

    import os
    import cv2
    import numpy as np
    from collections import defaultdict

    # -------------------------
    # Read file list
    # -------------------------
    all_files = list_files(folder)

    grouped = defaultdict(dict)
    labels = {}

    # -------------------------
    # Parse file names
    # -------------------------
    for file in all_files:

        # Extract label
        label_part = file.split("_")[0]  # drownsinessLevel0
        label = int(label_part.replace("drownsinessLevel", ""))

        # Extract channel
        for ch in channels:
            if f"channel{ch}" in file:
                channel = ch
                break
        else:
            continue  # skip if channel not found

        # Build sample_id (remove channel part)
        sample_id = file.replace(f"_channel{channel}", "")

        grouped[sample_id][channel] = file  #dict inside dict
        labels[sample_id] = label

    # -------------------------
    # Build X and Y
    # -------------------------
    X_list = []
    Y_list = []

    for sample_id in grouped:

        imgs = []

        for ch in channels:

            if ch not in grouped[sample_id]:
                break  # skip incomplete samples

            full_path = os.path.join(folder, grouped[sample_id][ch])

            if not os.path.exists(full_path):
                break
            
            if cmap=="viridis":
                img = cv2.imread(full_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            elif cmap=="gray":
                img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
                img = img[..., np.newaxis]
            
            img = img / 255.0

            imgs.append(img)

        if len(imgs) == len(channels):
            stacked = np.concatenate(imgs, axis=-1)
            X_list.append(stacked)
            Y_list.append(labels[sample_id])

    X = np.stack(X_list)
    Y = np.array(Y_list)
    Y = Y[:, np.newaxis]


    print("Final dataset shape:", X.shape)
    print("Labels shape:", Y.shape)

    return X, Y

Importing the images which will be used for training the model:

In [4]:
(X, y) = load_data(folder=config.DATA_DIR / "GeneratedScalogramsGrayOverlap0.733",
                   channels=["C3", "C4"], 
                   cmap="gray")

rus = RandomUnderSampler(random_state=master_seed)

rus.fit_resample(np.zeros(len(y)).reshape(-1,1), y)

indices = rus.sample_indices_

X = X[indices]
y = y[indices]

print(X.shape)
print(y.shape)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=master_seed)

print(y_train.shape)
print(y_test.shape)

Final dataset shape: (2592, 64, 64, 2)
Labels shape: (2592, 1)
(1584, 64, 64, 2)
(1584, 1)
(1108, 1)
(476, 1)


Building a CNN-2D model with Keras:

In [ ]:
model = Sequential()
model.add(Conv2D(64, (3,3), activation='relu', input_shape=(64,64,2))),
model.add(keras.layers.BatchNormalization(
                            momentum=master_momentum,
                            epsilon=master_epsilon,
                            center=True,
                            scale=True,
                            beta_initializer="zeros",
                            gamma_initializer="ones",
                            moving_mean_initializer="zeros",
                            moving_variance_initializer="ones",
                            )),
model.add(MaxPooling2D(2,2)),
model.add(Dropout(0.25, seed = master_seed)),
model.add(Conv2D(32, (3,3), activation='relu')),
model.add(keras.layers.BatchNormalization(
                            momentum=master_momentum,
                            epsilon=master_epsilon,
                            center=True,
                            scale=True,
                            beta_initializer="zeros",
                            gamma_initializer="ones",
                            moving_mean_initializer="zeros",
                            moving_variance_initializer="ones",
                            )),
model.add(MaxPooling2D(2,2)),
model.add(Dropout(0.25, seed = master_seed)),
model.add(Flatten()),
model.add(Dense(128, activation='relu')),
model.add(Dropout(0.5, seed = master_seed)),
## Temporarily, we are going to use a simple output without sigmoid function,
## which outputs a logit (-inf, inf). Later, we will use a SVM here.
## Using softmax here does not work, because keras.losses.BinaryCrossentropy
## expects only one y_pred float number, not two.
model.add(Dense(1))

model.summary()

Compiling the model:

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate = master_learning_rate)


metrics=[
        ## Thresholds are 0.0 because we are output logit
        keras.metrics.BinaryAccuracy(threshold=0.0, name="accuracy"),
        keras.metrics.AUC(num_thresholds=200, curve="ROC", from_logits=True, name="auc"),
        ]

model.compile(optimizer=optimizer,
            loss=keras.losses.BinaryCrossentropy(from_logits=True),
            metrics = metrics
            )

Training and validating the model:

In [ ]:
callback = keras.callbacks.EarlyStopping(
            monitor="val_loss",
            min_delta=0,
            patience=10,
            mode="min",
            restore_best_weights=False,
            start_from_epoch=0, 
)

history = model.fit(x = x_train, y = y_train,
                    epochs=50,
                    batch_size=master_batch_size, 
                    validation_data=(x_test, y_test),
                    callbacks=[callback],
                    )


print(history.history.keys())

Saving the trained model and the training history:

In [ ]:
if SAVE_MODEL:
    keras.models.save_model(model=model, filepath= config.OUTPUT_DIR / "KerasModels" /  f"{master_model_name}.keras" )

    with open( config.OUTPUT_DIR / "KerasModels" / f"{master_model_name}_history.json", "w") as f:
        json.dump(history.history, f)

Plotting data related to the history of training:

In [ ]:
metrics = [
    ("accuracy", "Accuracy"),
    ("loss", "Loss"),
    ("auc", "AUC"),
]

for key, title in metrics:
    plt.figure()
    plt.plot(history.history[key])
    plt.plot(history.history["val_" + key])
    plt.title(f"Model {title}")
    plt.ylabel(title)
    plt.xlabel("Epoch")
    plt.legend(["Train", "Validation"])
    plt.show()

Making some predictions:

In [ ]:
predictions = sigmoid( model.predict(x_test) ).numpy()
i = 30

logits = model.predict(x_test)
print(logits.min(), logits.max())

Finding all the wrong predictions:

In [ ]:
error_classification = []

for i in range(len(predictions)):
  if round(float(predictions[i][0])) != int(y_test[i][0]):
    error_classification.append(i)

print('All wrong predictions: ', len(error_classification))
print('List of wrong predictions: ', error_classification)
print(f'Final Accuracy: { ( 100*float( 1- len(error_classification)/len(predictions))) }')